In [1]:
!pip install nltk

In [2]:
import re
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()


[nltk_data] Downloading package punkt to /home/computer/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/computer/nltk_data...
[nltk_data] Downloading package omw-1.4 to /home/computer/nltk_data...


In [4]:
# ---------- Load dataset ----------
tickets_df = pd.read_csv("train.csv")

print("Columns:", tickets_df.columns.tolist())
print("Shape:", tickets_df.shape)


Columns: ['count', 'hate_speech_count', 'offensive_language_count', 'neither_count', 'class', 'tweet']
Shape: (24783, 6)


In [5]:
# ---------- Auto-detect the real text column ----------
candidate_cols = []
for col in tickets_df.columns:
    sample = tickets_df[col].dropna().astype(str).head(20)
    avg_len = sample.str.len().mean() if len(sample) > 0 else 0
    candidate_cols.append((col, avg_len))

candidate_cols.sort(key=lambda x: x[1], reverse=True)
print("\nColumns ranked by average string length:")
for col, avg_len in candidate_cols:
    print(f"  {col!r:30} avg_len={avg_len:.1f}")

text_col = candidate_cols[0][0]
print(f"\nAuto-selected text column: {text_col!r}")



Columns ranked by average string length:
  'tweet'                        avg_len=70.3
  'count'                        avg_len=1.0
  'hate_speech_count'            avg_len=1.0
  'offensive_language_count'     avg_len=1.0
  'neither_count'                avg_len=1.0
  'class'                        avg_len=1.0

Auto-selected text column: 'tweet'


In [6]:
# ---------- OVERRIDE HERE IF AUTO-DETECT PICKS THE WRONG COLUMN ----------
# text_col = "PUT_CORRECT_COLUMN_NAME_HERE"

ticket_text = str(tickets_df.loc[0, text_col])
print("\nRaw Ticket Text:\n", ticket_text)

if ticket_text.strip().isdigit() or len(ticket_text.strip()) < 5:
    print("\n⚠️  WARNING: ticket_text looks like a number/short code, not real text.")
    print("⚠️  Check the column list above and set text_col manually.")




Raw Ticket Text:
 !!! RT @mayasolovely: As a woman you shouldn't complain about cleaning up your house. &amp; as a man you should always take the trash out...


In [8]:
# Pre-step: basic filtration + tokenization (needed as input)
def filter_text(text):
    text = str(text)
    text = re.sub(r'http\S+|www\S+|\S+@\S+', '', text)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

filtered_ticket = filter_text(ticket_text)
ticket_words = word_tokenize(filtered_ticket)

print("\n--- Filtered Text (input to this experiment) ---")
print(filtered_ticket)
print("\nTokens:\n", ticket_words)


--- Filtered Text (input to this experiment) ---
RT mayasolovely As a woman you shouldn t complain about cleaning up your house amp as a man you should always take the trash out

Tokens:
 ['RT', 'mayasolovely', 'As', 'a', 'woman', 'you', 'shouldn', 't', 'complain', 'about', 'cleaning', 'up', 'your', 'house', 'amp', 'as', 'a', 'man', 'you', 'should', 'always', 'take', 'the', 'trash', 'out']


In [9]:
def remove_stopwords(words):
    return [w for w in words if w.lower() not in stop_words]

filtered_words = remove_stopwords(ticket_words)

print("\n--- Stop Word Removal ---")
print("Original Word Count:", len(ticket_words))
print("After Stop Word Removal:", len(filtered_words))
print("Tokens without stopwords:\n", filtered_words)



--- Stop Word Removal ---
Original Word Count: 25
After Stop Word Removal: 11
Tokens without stopwords:
 ['RT', 'mayasolovely', 'woman', 'complain', 'cleaning', 'house', 'amp', 'man', 'always', 'take', 'trash']


In [10]:
def stem_words(words):
    return [stemmer.stem(w) for w in words]

stemmed_words = stem_words(filtered_words)

print("\n--- Stemming (Porter Stemmer) ---")
print("Stemmed Tokens:\n", stemmed_words)


--- Stemming (Porter Stemmer) ---
Stemmed Tokens:
 ['rt', 'mayasolov', 'woman', 'complain', 'clean', 'hous', 'amp', 'man', 'alway', 'take', 'trash']


In [11]:
def lemmatize_words(words):
    return [lemmatizer.lemmatize(w, pos='v') for w in words]  # verb-aware lemmatization

lemmatized_words = lemmatize_words(filtered_words)

print("\n--- Lemmatization (WordNet Lemmatizer) ---")
print("Lemmatized Tokens:\n", lemmatized_words)



--- Lemmatization (WordNet Lemmatizer) ---
Lemmatized Tokens:
 ['RT', 'mayasolovely', 'woman', 'complain', 'clean', 'house', 'amp', 'man', 'always', 'take', 'trash']


In [12]:
comparison_df = pd.DataFrame({
    "Original": filtered_words,
    "Stemmed": stemmed_words,
    "Lemmatized": lemmatized_words
})

print("\n--- Stemming vs Lemmatization Comparison ---")
print(comparison_df)


--- Stemming vs Lemmatization Comparison ---
        Original    Stemmed    Lemmatized
0             RT         rt            RT
1   mayasolovely  mayasolov  mayasolovely
2          woman      woman         woman
3       complain   complain      complain
4       cleaning      clean         clean
5          house       hous         house
6            amp        amp           amp
7            man        man           man
8         always      alway        always
9           take       take          take
10         trash      trash         trash
